In [ ]:
import torch

def check_cuda_gpu():
    if torch.cuda.is_available():
        print(f"CUDA is available. Number of GPUs: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    else:
        print("CUDA is not available.")

check_cuda_gpu()

: 

In [ ]:
import time

import torch.nn as nn

def run_operations(device, mat_size=1024):
    dev = torch.device(device)
    print(f"\n--- Running ops on {dev} ---")
    torch.manual_seed(0)

    # Matrix multiplication
    a = torch.randn(mat_size, mat_size, device=dev)
    b = torch.randn(mat_size, mat_size, device=dev)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    c = torch.matmul(a, b)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    print(f"matmul: {t1 - t0:.4f}s, result shape: {c.shape}")

    # Elementwise ops + reduction
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    r = (a * b).relu().sum()
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    print(f"elementwise+reduce: {t1 - t0:.4f}s, scalar result: {r.item():.4e}")

    # Small conv forward (simulates NN workload)
    x = torch.randn(8, 3, 224, 224, device=dev)
    conv = nn.Conv2d(3, 16, kernel_size=3, padding=1).to(dev)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    y = conv(x)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    print(f"conv forward: {t1 - t0:.4f}s, output shape: {y.shape}")

# Decide devices to test
devices = ["cpu"]
if torch.cuda.is_available():
    devices.append("cuda:0")

# Use smaller sizes on CPU to avoid long runs
for d in devices:
    size = 1024 if ("cuda" in d and torch.cuda.is_available()) else 512
    run_operations(d, mat_size=size)